# RQ3 starter — Incremental predictive value of early transitions

Starter notebook generated from the finalized RQ1 handoff. It keeps BKT/PFA as explicit placeholders rather than inventing their implementation.

This starter loads the frozen RQ1 handoff: `rq1_merged_spm_quade_outputs/rq1_objects_for_rq2_rq3.pkl`.


# RQ3 starter — Incremental predictive value of early transitions

**Research question:** Do early transition features add predictive value for later struggle beyond baseline correctness, assistance counts, BKT, and PFA?

### Planned nested-model logic

The core comparison should be **incremental prediction**, not simply whether transition coefficients are individually significant.

- **Baseline model:** early correctness / baseline correctness features.
- **+ Assistance:** add assistance-count features.
- **+ BKT:** add the pre-specified BKT knowledge estimate(s).
- **+ PFA:** add the pre-specified PFA feature(s).
- **+ Early transitions:** add the frozen 49 transition indicators/counts.

The final RQ3 analysis should compare the transition-augmented model against the strongest non-transition baseline using a pre-specified metric such as held-out log loss/AUC and an appropriate nested-model comparison where applicable.

**BKT and PFA are not fabricated here.** The starter code provides explicit placeholders for the user's finalized BKT/PFA outputs or implementations.


In [ ]:
# ================================================================
# RQ3 STARTER CODE
# ================================================================
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression

HANDOFF_PATH = Path("rq1_merged_spm_quade_outputs/rq1_objects_for_rq2_rq3.pkl")

with open(HANDOFF_PATH, "rb") as f:
    handoff = pickle.load(f)

population = handoff["opt_population"].copy()
transition_counts = handoff["opt_transition_counts"].copy()
selected_k = handoff["selected_k"]
ALL_TRANSITIONS = handoff["all_transitions"]

# One row per student × skill history.
base = population[
    [
        "userId", "skill", "later_struggle",
        "early_score", "early_scaffold_share",
        "early_algebra_share", "early_choose_1_share",
        "early_fill_in_1_share", "mean_early_item_easiness"
    ]
].copy()

# Add early transition counts.
base = base.merge(
    transition_counts[["userId", "skill"] + ALL_TRANSITIONS],
    on=["userId", "skill"],
    how="left"
)

# ------------------------------------------------
# RQ3.1 Baseline predictors
# ------------------------------------------------
# This is the minimal baseline starter. Add the finalized
# correctness/assistance specification before the final run.
baseline_cols = [
    "early_score",
    "early_scaffold_share",
    "early_algebra_share",
    "early_choose_1_share",
    "early_fill_in_1_share",
    "mean_early_item_easiness",
]

X_base = base[baseline_cols].fillna(0).astype(float)
y = base["later_struggle"].astype(int)

# ------------------------------------------------
# RQ3.2 Assistance-count block
# ------------------------------------------------
# TODO: replace/add the exact assistance-count variables specified
# in the final RQ3 protocol. Keep these separate from the transition
# block so incremental value is identifiable.
assistance_cols = [
    # e.g. "early_hint_count", "early_bottom_hint_count",
    # "early_scaffold_count"
]

# ------------------------------------------------
# RQ3.3 BKT block
# ------------------------------------------------
# TODO: merge finalized BKT estimates at the same student × skill
# prediction unit. Example placeholder:
bkt_cols = [
    # "bkt_p_known_end_early"
]

# ------------------------------------------------
# RQ3.4 PFA block
# ------------------------------------------------
# TODO: merge finalized PFA features at the same prediction unit.
pfa_cols = [
    # "pfa_logit_early"
]

# ------------------------------------------------
# RQ3.5 Transition block
# ------------------------------------------------
transition_cols = ALL_TRANSITIONS

X_transition = base[transition_cols].fillna(0).astype(float)

# Example nested design matrices.
X0 = X_base.copy()
X1 = pd.concat([X0, base[[c for c in assistance_cols if c in base.columns]]], axis=1)
X2 = pd.concat([X1, base[[c for c in bkt_cols if c in base.columns]]], axis=1)
X3 = pd.concat([X2, base[[c for c in pfa_cols if c in base.columns]]], axis=1)
X4 = pd.concat([X3, X_transition], axis=1)

print("RQ3 prediction unit:", base[["userId", "skill"]].drop_duplicates().shape[0])
print("Selected k:", selected_k)
print("Baseline predictors:", list(X0.columns))
print("Transition predictors:", len(transition_cols))
print("BKT/PFA placeholders still require finalized variables:",
      bkt_cols, pfa_cols)

# ------------------------------------------------
# RQ3.6 Starter: grouped cross-validated comparison
# ------------------------------------------------
# IMPORTANT: choose the grouping unit and CV scheme before final inference.
# The example below groups by student to avoid placing histories from the
# same student in both train and test folds.
groups = base["userId"]

def cv_auc(model, X, y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    pred = cross_val_predict(
        model, X, y, cv=gkf, groups=groups, method="predict_proba"
    )[:, 1]
    return roc_auc_score(y, pred), log_loss(y, pred)

model = LogisticRegression(max_iter=2000, solver="liblinear")

auc0, ll0 = cv_auc(model, X0, y, groups)
auc4, ll4 = cv_auc(model, X4, y, groups)

comparison = pd.DataFrame({
    "model": ["baseline", "baseline_plus_early_transitions"],
    "AUC": [auc0, auc4],
    "log_loss": [ll0, ll4],
    "delta_AUC_vs_baseline": [0.0, auc4 - auc0],
    "delta_log_loss_vs_baseline": [0.0, ll4 - ll0],
})

display(comparison)

print(
    "RQ3 starter complete. Before final inference, finalize the assistance, "
    "BKT and PFA feature definitions and pre-specify the evaluation/comparison protocol."
)
